# Compare Predicted vs Observed NO₂ at AirNow Stations

This notebook reads `predictions.nc` and produces:
1. **Side-by-side cartopy maps** of 3-month-mean predicted and observed NO₂
2. **Animated GIF** stepping through every time step with predicted vs observed maps

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import imageio.v3 as iio
from io import BytesIO

REPO_ROOT = Path('/mnt/data3/GraphMamba')
NC_PATH = REPO_ROOT / 'output' / 'test' / 'predictions.nc'
OUT_DIR = REPO_ROOT / 'output' / 'test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

ds = xr.open_dataset(NC_PATH, decode_timedelta=False)
print(ds)

In [ ]:
lat = ds['latitude'].values
lon = ds['longitude'].values
has_obs = ds['has_observation'].values  # (time, site)
pred = ds['no2'].values                 # (time, site)
obs  = ds['observed_no2'].values        # (time, site)
times = ds['time'].values

# Mask out entries with no observation
obs_masked = np.where(has_obs > 0.5, obs, np.nan)
pred_masked = np.where(has_obs > 0.5, pred, np.nan)

# 3-month mean per site (only where observations exist)
with np.errstate(all='ignore'):
    mean_obs  = np.nanmean(obs_masked, axis=0)
    mean_pred = np.nanmean(pred_masked, axis=0)

print(f'Sites: {len(lat)}, Time steps: {len(times)}')
print(f'Mean obs range:  {np.nanmin(mean_obs):.2f} – {np.nanmax(mean_obs):.2f} ppb')
print(f'Mean pred range: {np.nanmin(mean_pred):.2f} – {np.nanmax(mean_pred):.2f} ppb')

In [ ]:
# ── Shared colormap settings ──
VMIN, VMAX = 0, max(np.nanmax(mean_obs), np.nanmax(mean_pred))
CMAP = 'RdYlGn_r'
PROJ = ccrs.LambertConformal(central_longitude=-112, central_latitude=42)
PAD = 2
EXTENT = [lon.min() - PAD, lon.max() + PAD, lat.min() - PAD, lat.max() + PAD]

def make_panel(ax, lons, lats, values, title, vmin=VMIN, vmax=VMAX, cmap=CMAP):
    """Draw a single cartopy panel with circle markers."""
    ax.set_extent(EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#f0f0f0')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    valid = ~np.isnan(values)
    sc = ax.scatter(
        lons[valid], lats[valid], c=values[valid],
        s=40, cmap=cmap, vmin=vmin, vmax=vmax,
        edgecolors='k', linewidths=0.3,
        transform=ccrs.PlateCarree(), zorder=5,
    )
    ax.set_title(title, fontsize=11, fontweight='bold')
    return sc

print('Helper ready')

## 1 · Three-Month Mean Maps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6),
                         subplot_kw={'projection': PROJ})

sc1 = make_panel(axes[0], lon, lat, mean_pred, 'Predicted NO₂ (3-month mean)')
sc2 = make_panel(axes[1], lon, lat, mean_obs,  'Observed NO₂ (3-month mean)')

cbar = fig.colorbar(sc2, ax=axes, orientation='horizontal',
                    fraction=0.04, pad=0.07, shrink=0.6)
cbar.set_label('NO₂ (ppb)', fontsize=11)

fig.suptitle('Predicted vs Observed NO₂ at AirNow Stations\n'
             'Jul – Sep 2024 Mean', fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()

mean_png = OUT_DIR / 'no2_mean_comparison.png'
fig.savefig(mean_png, dpi=180, bbox_inches='tight')
print(f'Saved → {mean_png}')
plt.show()

## 2 · Animated GIF (Predicted vs Observed over Time)

In [ ]:
# Use a fixed vmin/vmax for the animation across all frames
ANIM_VMAX = 40  # ppb – clip for better contrast

# Sample every Nth time step to keep GIF size reasonable
STEP = max(1, len(times) // 200)  # ~200 frames max
time_indices = list(range(0, len(times), STEP))
print(f'Total time steps: {len(times)}, sampling every {STEP} → {len(time_indices)} frames')

frames = []
for idx, ti in enumerate(time_indices):
    t_label = str(times[ti])[:19]
    p = pred[ti]
    o = obs_masked[ti]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                             subplot_kw={'projection': PROJ})
    make_panel(axes[0], lon, lat, p, f'Predicted NO₂', vmax=ANIM_VMAX)
    sc = make_panel(axes[1], lon, lat, o, f'Observed NO₂', vmax=ANIM_VMAX)

    cbar = fig.colorbar(sc, ax=axes, orientation='horizontal',
                        fraction=0.04, pad=0.07, shrink=0.6)
    cbar.set_label('NO₂ (ppb)', fontsize=10)
    fig.suptitle(f'{t_label} UTC', fontsize=12, fontweight='bold', y=1.0)
    fig.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    frames.append(iio.imread(buf))

    if (idx + 1) % 50 == 0:
        print(f'  rendered {idx + 1}/{len(time_indices)} frames')

gif_path = OUT_DIR / 'no2_comparison.gif'
iio.imwrite(gif_path, frames, duration=150, loop=0)
print(f'\nSaved GIF → {gif_path}  ({len(frames)} frames)')

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(gif_path)))